# Revision R4 — second detector family (MLP, FT-Transformer), binary task. POST HOC.
Run N0 → N1 → N2 → N3 → N4 in order on a GPU runtime (A100). Nothing frozen is modified; outputs go to `revision_R4/`.

### CELL N0 — setup, custom layers, model loading, reproduction gates

In [ ]:
# CELL N0 — setup, custom layers, model loading, reproduction gates
from google.colab import drive
drive.mount("/content/drive")
import os, gc, json, glob, time, shutil
import numpy as np, pandas as pd, tensorflow as tf, keras
BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2 = f"{BASE}/V2_adaptive_reliability"
OUT = f"{V2}/revision_R4"; os.makedirs(OUT, exist_ok=True)
WINDOW, EPS = 5000, 0.01
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

# --- custom layers, verbatim from V1 Step 6.3D (needed to deserialise the FT-Transformer) ---
@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class NumericalFeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, n_features, token_dim, **kwargs):
        super().__init__(**kwargs); self.n_features = int(n_features); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.feature_weight = self.add_weight(name="feature_weight", shape=(self.n_features, self.token_dim), initializer="glorot_uniform", trainable=True)
        self.feature_bias = self.add_weight(name="feature_bias", shape=(self.n_features, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.expand_dims(inputs, axis=-1) * self.feature_weight + self.feature_bias
    def get_config(self):
        c = super().get_config(); c.update({"n_features": self.n_features, "token_dim": self.token_dim}); return c

@tf.keras.utils.register_keras_serializable(package="TrustShiftIDS")
class CLSToken(tf.keras.layers.Layer):
    def __init__(self, token_dim, **kwargs):
        super().__init__(**kwargs); self.token_dim = int(token_dim)
    def build(self, input_shape):
        self.cls_token = self.add_weight(name="cls_token", shape=(1, 1, self.token_dim), initializer="zeros", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        return tf.concat([tf.tile(self.cls_token, [tf.shape(inputs)[0], 1, 1]), inputs], axis=1)
    def get_config(self):
        c = super().get_config(); c.update({"token_dim": self.token_dim}); return c

CUSTOM = {"NumericalFeatureTokenizer": NumericalFeatureTokenizer, "CLSToken": CLSToken,
          "TrustShiftIDS>NumericalFeatureTokenizer": NumericalFeatureTokenizer, "TrustShiftIDS>CLSToken": CLSToken}
MODELS = {"MLP": f"{BASE}/models/mlp/MLP_binary_seed42.keras",
          "FTTransformer": f"{BASE}/models/ft_transformer/FTTransformer_binary_seed42.keras"}
M = {}
for name, p in MODELS.items():
    M[name] = keras.models.load_model(p, compile=False, custom_objects=CUSTOM)
    print(f"[N0] loaded {name}: input {M[name].input_shape} output {M[name].output_shape}")

# --- frozen neural preprocessing (V1 Step 5.5C): clip + standardise continuous features only ---
SPEC = pd.read_csv(f"{BASE}/metadata/TrustShift_IDS_FROZEN_NEURAL_PREPROCESSING_SPEC.csv")
assert len(SPEC) == 68
FEATURES = SPEC["feature_id"].tolist()
CONT = SPEC[SPEC.feature_group == "continuous"]
assert len(CONT) == 56
CONT_IDX = np.array([FEATURES.index(f) for f in CONT.feature_id]); LO = CONT.lower_bound.to_numpy(float); HI = CONT.upper_bound.to_numpy(float)
MU = CONT["mean"].to_numpy(float); SD = CONT["std"].to_numpy(float)
def neural_transform(X):
    X = np.array(X, dtype=np.float64, copy=True)
    X[:, CONT_IDX] = (np.clip(X[:, CONT_IDX], LO, HI) - MU) / SD
    return X

def p_attack(model, Xn, bs=16384):
    out = model.predict(Xn.astype(np.float32), batch_size=bs, verbose=0)
    return np.asarray(out, dtype=np.float64).reshape(-1)

# Gate G1: transform(IID imputed) must equal stored IID neural representation
P_IMP = f"{BASE}/data/processed/CICIDS2017_68F_imputed"; P_NEU = f"{BASE}/data/processed/CICIDS2017_68F_neural"
iid_imp = pd.read_parquet(f"{P_IMP}/CICIDS2017_iid_test_68F_imputed.parquet")
iid_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_iid_test_68F_neural.parquet", columns=FEATURES)
d = float(np.max(np.abs(neural_transform(iid_imp[FEATURES].to_numpy()) - iid_neu.to_numpy(np.float64))))
print(f"[N0] G1 transform reproduction: max|diff| = {d:.3e}", "PASS" if d < 1e-9 else "FAIL"); assert d < 1e-9
raw = iid_imp["binary_label"]
y_iid = raw.to_numpy(np.int64) if np.issubdtype(raw.dtype, np.number) else raw.astype(str).str.lower().map({"benign": 0, "attack": 1, "0": 0, "1": 1}).to_numpy(np.int64)
X_iid_raw = iid_imp[FEATURES].to_numpy(np.float64); del iid_imp, iid_neu; gc.collect()

# Gate G2: reproduce the saved V1 validation predictions
val_neu = pd.read_parquet(f"{P_NEU}/CICIDS2017_validation_68F_neural.parquet", columns=FEATURES).to_numpy(np.float64)
for name in M:
    sp = f"{BASE}/results/{'mlp' if name=='MLP' else 'ft_transformer'}/{name}_binary_seed42_validation_predictions.parquet"
    saved = pd.read_parquet(sp); print(f"[N0] saved {name} validation columns: {list(saved.columns)}")
    cand = [c for c in saved.columns if any(k in c.lower() for k in ["prob", "p1", "p_attack", "score", "proba"])]
    p_now = p_attack(M[name], val_neu)
    if cand:
        dd = float(np.max(np.abs(p_now - saved[cand[0]].to_numpy(np.float64))))
        print(f"[N0] G2 {name}: column '{cand[0]}' max|diff| = {dd:.3e}", "PASS" if dd < 1e-4 else "CHECK")
    else:
        print(f"[N0] G2 {name}: no probability column recognised — paste the column list above")
del val_neu; gc.collect()
print("[N0] COMPLETE")

### CELL N1 — temperature scaling on calibration, thresholds, ATC, IID test, 54 source-null windows

In [ ]:
# CELL N1 — temperature scaling on calibration, thresholds, ATC, IID test, 54 source-null windows
from scipy.optimize import minimize_scalar
cal = pd.read_parquet(f"{P_NEU}/CICIDS2017_calibration_68F_neural.parquet")
assert len(cal) == 274787
Xc = cal[FEATURES].to_numpy(np.float64)
rc = cal["binary_label"]; yc = rc.to_numpy(np.int64) if np.issubdtype(rc.dtype, np.number) else rc.astype(str).str.lower().map({"benign":0,"attack":1,"0":0,"1":1}).to_numpy(np.int64)
del cal; gc.collect()
COV_GRID = [0.999, 0.995, 0.99, 0.98, 0.975, 0.95, 0.925, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
def logit(p): p = np.clip(p, 1e-12, 1 - 1e-12); return np.log(p) - np.log1p(-p)
def calibrated(p, T): return 1.0 / (1.0 + np.exp(-logit(p) / T))
def risk_pred(p, T):
    q = calibrated(p, T); pred = (q >= 0.5).astype(np.int64); return 1.0 - np.maximum(q, 1 - q), pred
CAL = {}
for name, model in M.items():
    p = p_attack(model, Xc)
    nll = lambda logT: float(np.mean(-(yc * np.log(np.clip(calibrated(p, np.exp(logT)), 1e-15, 1)) + (1 - yc) * np.log(np.clip(1 - calibrated(p, np.exp(logT)), 1e-15, 1)))))
    T = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))
    r, pr = risk_pred(p, T); err = pr != yc
    # fixed threshold: loosest threshold whose calibration selective risk <= 1%
    order = np.argsort(r, kind="mergesort"); cum_err = np.cumsum(err[order]); k = np.arange(1, len(r) + 1)
    ok = np.where(cum_err / k <= EPS)[0]; tau_fixed = float(r[order][ok.max()])
    thr = {"fixed": tau_fixed}
    rs = np.sort(r)
    for c in [0.99, 0.90]: thr[{0.99: "elevated", 0.90: "severe"}[c]] = float(rs[int(np.ceil(c * len(r))) - 1])
    for c in COV_GRID: thr[f"cov_{c:.3f}"] = float(rs[int(np.ceil(c * len(r))) - 1])
    conf = 1 - r; e_src = float(err.mean()); atc_t = float(np.quantile(conf, e_src))
    perm = np.random.default_rng(42).permutation(len(r))[:54 * WINDOW].reshape(54, WINDOW)
    null_atc = []
    for w in range(54):
        idx = perm[w]; rest = np.ones(len(r), bool); rest[idx] = False
        null_atc.append(float(np.mean(conf[idx] < np.quantile(conf[rest], float(err[rest].mean())))))
    atc_b = [float(np.quantile(null_atc, 0.95)), float(np.quantile(null_atc, 0.99))]
    ri, pi = risk_pred(p_attack(model, neural_transform(X_iid_raw)), T); ei = pi != y_iid
    CAL[name] = {"T": T, "thresholds": thr, "atc_conf_threshold": atc_t, "atc_boundaries": atc_b, "cal_errors": int(err.sum()),
                 "cal_cov_fixed": float(np.mean(r <= tau_fixed)), "iid_errors": int(ei.sum()), "iid_accuracy": float(1 - ei.mean()),
                 "iid_fixed_coverage": float(np.mean(ri <= tau_fixed)),
                 "iid_fixed_risk": float(ei[ri <= tau_fixed].mean()) if (ri <= tau_fixed).any() else None}
    print(f"[N1] {name}: T={T:.4f} | cal errors {err.sum()} | fixed tau {tau_fixed:.4g} (cal cov {CAL[name]['cal_cov_fixed']:.4f}) | "
          f"IID errors {ei.sum()} acc {CAL[name]['iid_accuracy']:.5f} | ATC bounds {atc_b}")
json.dump(CAL, open(f"{OUT}/N1_neural_calibration_and_thresholds.json", "w"), indent=2)
del Xc; gc.collect(); print("[N1] COMPLETE")

### CELL N2 — deployment sequences (same five orderings and features as the XGBoost analysis)

In [ ]:
# CELL N2 — deployment sequences (same five orderings and features as the XGBoost analysis)
def window_rows(r, pred, y, base, name):
    rows = []; conf = 1 - r; att = y != 0; err = pred != y
    atc = float(np.mean(conf < CAL[name]["atc_conf_threshold"]))
    for tn, t in CAL[name]["thresholds"].items():
        acc = r <= t
        rows.append({**base, "detector": name, "thr_name": tn, "threshold": t, "atc_est": atc, "n": len(y), "acc_n": int(acc.sum()),
                     "acc_err": int((acc & err).sum()), "n_err": int(err.sum()), "n_attack": int(att.sum()), "acc_attack": int((acc & att).sum()),
                     "acc_missed_attack": int((acc & att & (pred == 0)).sum()), "acc_benign": int((acc & ~att).sum()),
                     "acc_false_alarm": int((acc & ~att & (pred == 1)).sum())})
    return rows
mon = pd.read_csv(f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13C_MULTI_SEED_LABEL_FREE_MONITORING_WINDOWS.csv")
rows = []
for seed in [1, 7, 21, 42, 84]:
    z = np.load(f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{seed}_FLOAT64_STREAM_FEATURES.npz", allow_pickle=False)
    assert z["feature_order"].tolist() == FEATURES
    Xs = neural_transform(z["X"]); src = np.asarray(z["source_row_index"], np.int64); z.close(); y = y_iid[src]
    ms = mon[mon.deployment_order_seed == seed].sort_values("global_window_id").reset_index(drop=True)
    for name, model in M.items():
        r, pr = risk_pred(p_attack(model, Xs), CAL[name]["T"])
        for w in range(60):
            a, b = w * WINDOW, (w + 1) * WINDOW
            rows += window_rows(r[a:b], pr[a:b], y[a:b], {"seed": seed, "window_id": w, "stage_name": ms.loc[w, "stage_name"],
                                                          "xgb_sss_state": str(ms.loc[w, "primary_shift_evidence_state"]).lower()}, name)
    del Xs; gc.collect(); print(f"  seed {seed}: done")
pd.DataFrame(rows).to_csv(f"{OUT}/N2_neural_stream_window_outcomes.csv.gz", index=False); print("[N2] COMPLETE", len(rows))

### CELL N3 — cross-dataset transfer to CSE-CIC-IDS2018 (same file order as the XGBoost shards)

In [ ]:
# CELL N3 — cross-dataset transfer to CSE-CIC-IDS2018 (same file order as the XGBoost shards)
import pyarrow.parquet as pq
EXT = f"{BASE}/data/processed/CSE_CIC_IDS2018_68F_imputed"
parts = sorted(glob.glob(f"{EXT}/part_*.parquet")); print("[N3] parts:", [os.path.basename(p) for p in parts])
shards = sorted(glob.glob(f"{BASE}/results/shift_evaluation/S1_external/predictions/Step_12_2B_S1_part_*.parquet"))
assert len(parts) == len(shards) == 10
print("[N3] external part columns (first):", pq.ParquetFile(parts[0]).schema.names[:12], "...")
R = {n: [] for n in M}; P = {n: [] for n in M}; Y = []; SH = []
for i, (pp, sp) in enumerate(zip(parts, shards)):
    df = pd.read_parquet(pp); ref = pd.read_parquet(sp, columns=["binary_true"])
    lab = [c for c in ["binary_label", "binary_true", "label_binary"] if c in df.columns][0]
    yy = df[lab].to_numpy(np.int64) if np.issubdtype(df[lab].dtype, np.number) else df[lab].astype(str).str.lower().map({"benign":0,"attack":1,"0":0,"1":1}).to_numpy(np.int64)
    assert len(yy) == len(ref) and np.array_equal(yy, ref["binary_true"].to_numpy(np.int64)), f"alignment failure in part {i+1}"
    Xn = neural_transform(df[FEATURES].to_numpy(np.float64)); del df, ref; gc.collect()
    for name, model in M.items():
        r, pr = risk_pred(p_attack(model, Xn, bs=65536), CAL[name]["T"]); R[name].append(r.astype(np.float64)); P[name].append(pr.astype(np.int8))
    Y.append(yy.astype(np.int8)); SH.append(np.full(len(yy), i + 1, np.int16)); del Xn; gc.collect()
    print(f"  part {i+1}: {len(yy):,} flows aligned with XGBoost shard labels")
Y = np.concatenate(Y).astype(np.int64); SH = np.concatenate(SH); N = len(Y); assert N == 16_232_943
rows = []
for name in M:
    r = np.concatenate(R[name]); pr = np.concatenate(P[name]).astype(np.int64)
    print(f"[N3] {name}: accuracy {np.mean(pr == Y):.4f} | attack recall {np.mean(pr[Y == 1] == 1):.4f} | benign specificity {np.mean(pr[Y == 0] == 0):.4f}")
    for w in range(N // WINDOW):
        a, b = w * WINDOW, (w + 1) * WINDOW
        rows += window_rows(r[a:b], pr[a:b], Y[a:b], {"window_id": w, "first_shard": int(SH[a])}, name)
pd.DataFrame(rows).to_csv(f"{OUT}/N3_neural_external_window_outcomes.csv.gz", index=False)
del R, P; gc.collect(); print("[N3] COMPLETE", len(rows))

### CELL N4 — package for upload

In [ ]:
# CELL N4 — package for upload
z = shutil.make_archive(f"{OUT}/TrustShift_revision_R4_export", "zip", OUT)
print("Upload:", z, f"({os.path.getsize(z)/1e6:.1f} MB)")